In [3]:
import pandas as pd
import numpy as np

def load_and_combine_data(file_path_1, file_path_2):
    print("Loading and combining datasets...")
    df1 = pd.read_csv(file_path_1, encoding='latin1')
    df2 = pd.read_csv(file_path_2, encoding='latin1')
    
    # Combine both years into a single dataframe
    df_combined = pd.concat([df1, df2], ignore_index=True)
    print(f"Combined dataset shape: {df_combined.shape[0]:,} rows.")
    return df_combined

def clean_retail_pipeline(df):
    print("\nStarting basic cleaning...")
    df_clean = df.copy()
    
    # 1. Handle Column Name Variations
    col_customer = 'Customer ID' if 'Customer ID' in df_clean.columns else 'CustomerID'
    col_price = 'Price' if 'Price' in df_clean.columns else 'UnitPrice'
    
    # 2. Drop Exact Duplicates & Missing Values
    df_clean.drop_duplicates(inplace=True)
    df_clean.dropna(subset=[col_customer, 'Description'], inplace=True)
    
    # 3. Format Data Types
    df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
    # Convert Customer ID to string without the .0 float formatting
    df_clean[col_customer] = df_clean[col_customer].astype(int).astype(str)
    # Standardize string formatting for Description
    df_clean['Description'] = df_clean['Description'].astype(str).str.upper().str.strip()
    
    # 4. Filter Anomalies (Returns, Free Items, System Codes)
    df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean[col_price] > 0)]
    invalid_prefixes = ('ADJUST', 'TEST')
    df_clean = df_clean[~df_clean['StockCode'].str.startswith(invalid_prefixes)]
    invalid_codes = ['C2', 'POST', 'D', 'M', 'PADS', 'DOT', 'CRUK', 'BANK CHARGES', 'AMAZONFEE']
    df_clean = df_clean[~df_clean['StockCode'].astype(str).str.upper().isin(invalid_codes)]
    
    print("\nStarting advanced product standardization (Latest Transaction Method)...")
    
    # 5. Master Data Management: Standardize Descriptions to Latest Entry
    # Sort chronologically
    df_clean = df_clean.sort_values(by='InvoiceDate', ascending=True)
    
    # Extract the most recent description for every StockCode
    latest_descriptions = df_clean.drop_duplicates(subset=['StockCode'], keep='last')
    description_mapping = latest_descriptions.set_index('StockCode')['Description'].to_dict()
    
    # Map the latest description back to all historical transactions
    df_clean['Description'] = df_clean['StockCode'].map(description_mapping)
    
    # Remove BOM
    df_clean.rename(columns=lambda x: x.replace('ï»¿', '').strip(), inplace=True)
    # Jika ada kolom 'Invoice' kosong di ujung kanan akibat pergeseran delimiter, hapus juga:
    if 'Invoice' in df_clean.columns and df_clean.columns.duplicated().any():
        df_clean = df_clean.loc[:, ~df_clean.columns.duplicated()]

    # Sort by date
    df_clean = df_clean.sort_values(by='InvoiceDate').reset_index(drop=True)
    
    print("\n" + "="*50)
    print("PIPELINE COMPLETE")
    print("="*50)
    print(f"Final cleaned dataset shape: {df_clean.shape[0]:,} rows.")
    
    return df_clean

In [4]:
# ==========================================
# EXECUTION BLOCK (Update paths if necessary)
# ==========================================
# Kaggle typically stores datasets in /kaggle/input/dataset-name/
path_2009 = '/kaggle/input/datasets/felixoctavius/online-retail-ii/online_retail_II_2009_2010.csv'
path_2010 = '/kaggle/input/datasets/felixoctavius/online-retail-ii/online_retail_II_2010_2011.csv'

# 1. Combine
df_raw = load_and_combine_data(path_2009, path_2010)

# 2. Clean
df_final = clean_retail_pipeline(df_raw)
   
# 3. Preview
display(df_final.head())

Loading and combining datasets...
Combined dataset shape: 1,067,371 rows.

Starting basic cleaning...

Starting advanced product standardization (Latest Transaction Method)...

PIPELINE COMPLETE
Final cleaned dataset shape: 790,704 rows.


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET POT,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


In [5]:
df_final.to_csv('/kaggle/working/cleaned.csv', index=False)